<a href="https://colab.research.google.com/github/ozair247/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozair247/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: scoring / ranking** — with a classification model as one ingredient, not the final output.

- My lane (Lane 2, from `w01_research_question.ipynb`) is: *"Which pages should be reviewed first for refresh?"*
- That question is **"which ones first"**, not **"is this one declining, yes/no."** The framing skill maps "which ones first" questions to ranking/scoring, and "yes/no" questions to classification.
- Proof it can't just be classification: the code cell below shows 16,262 of 30,000 pages (54%) already carry a "declining" flag. A yes/no label alone hands an editor a list of 16,262 pages with no order — useless when the team can only act on ~50 at a time. What the editor actually needs is those pages **ordered**, best-candidate-first.
- Under the hood I will still train a classifier (decline probability) — that's a normal building block, matching the starter pipeline's `03_train_model.py`. But the classifier's probability is an ingredient that gets turned into a rank, not the final deliverable. The deliverable is a ranked queue, so I'm naming the task **scoring / ranking**.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized .csv")

print("Rows:", len(df))
print()
print("trend_direction breakdown:")
print(df["trend_direction"].value_counts())
print()
declining_count = (df["trend_direction"] == "down").sum()
print(f"Pages flagged 'declining': {declining_count} of {len(df)} "
      f"({declining_count/len(df):.1%})")
print("-> A plain yes/no flag on that many pages is not a work queue. It needs an order.")

Rows: 30000

trend_direction breakdown:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Pages flagged 'declining': 16262 of 30000 (54.2%)
-> A plain yes/no flag on that many pages is not a work queue. It needs an order.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label`** = 1 when `trend_direction == "down"`, else 0.

- Source: a **defined rule**, not a genuinely future-observed outcome. `trend_direction` is computed by comparing `impressions_last_30d` against `impressions_prev_30d` inside the *same* 90-day snapshot (down = more than a 20% drop). Both windows already happened before I see the row.
- Honest problem with this: the framing skill's rule is "the target must be observed, not defined — a label from a rule means the model learns the rule, not the world." This label is closer to the "defined rule" side. It tells me a page already dropped, not that it is *about to* drop. A model trained on it will partly just be re-deriving `trend_pct` maths, and the data dictionary itself flags `trend_pct` / `trend_direction` as "label source — never a feature," which only works if I keep them fully separate from the inputs.
- I'm using it anyway for this starter task because it's what the runnable pipeline (`02_baseline_score.py`, `03_train_model.py`) already ships and validates against (see `outputs/model_report.md`). I'm not presenting it as the ideal capstone target.
- **Stronger version for later weeks:** a true future-window label — e.g. features from the first 60 days, label = decline over the next 30 — so the model predicts forward instead of describing the past. That's flagged directly in `docs/ml-intern-dataset-and-lane-guide.md` (section 5) as the upgrade path.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

base_rate = df["is_declining_label"].mean()
print(f"is_declining_label positive rate: {base_rate:.1%}")
print()
print("How the label is built, in the columns themselves (first 5 rows):")
display(df[["content_id", "impressions_last_30d", "impressions_prev_30d",
            "trend_pct", "trend_direction", "is_declining_label"]].head())


is_declining_label positive rate: 54.2%

How the label is built, in the columns themselves (first 5 rows):


,content_id,impressions_last_30d,impressions_prev_30d,trend_pct,trend_direction,is_declining_label
0,content_304f48230142,578,987,-41.4,down,1
1,content_a1fb4e703a9e,2501,5915,-57.7,down,1
2,content_9aa793d4d895,2382,6089,-60.9,down,1
3,content_331d6c4de07b,3626,4206,-13.8,stable,0
4,content_d99b7a2d90ca,4211,6452,-34.7,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50** — of the top 50 pages my ranking puts first, how many actually carry the decline label.

- Why this and not accuracy: 54.2% of all pages are already labeled "declining" (see Section 2 code). A model that just guesses "declining" for everyone is ~54% accurate and completely useless — accuracy would look fine while hiding a bad ranking. Precision@50 only asks about the pages an editor will actually touch.
- Why 50 and not 20 or 200: it matches review capacity language used in the lane guide ("Precision@50 if the team can act on 50 candidates") and it's the same K the starter pipeline reports in `outputs/model_report.md`, so my number stays comparable to the baseline already in this repo.
- What "good" means, stated before I train anything: **beat 0.542** (the base rate — see code below), because 0.542 is what I'd get by picking 50 random pages. The published starter numbers (baseline rule 0.240, random forest 0.740, in `docs/ml-intern-dataset-and-lane-guide.md`) show the bar moves a lot depending on method — but 0.542 is my honest floor, not 0.240, because a rule that scores *below* random guessing (see Section 5) isn't a real floor.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.

random_guess_precision = base_rate
print(f"Precision@50 target to beat (base rate / random guess): {random_guess_precision:.3f}")
print(f"That means: out of 50 randomly picked pages, about "
      f"{random_guess_precision*50:.0f} would already carry the decline label by chance.")
print("Success = my ranked top 50 clears this number by a real, checkable margin.")


Precision@50 target to beat (base rate / random guess): 0.542
That means: out of 50 randomly picked pages, about 27 would already carry the decline label by chance.
Success = my ranked top 50 clears this number by a real, checkable margin.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page, for one client, over a trailing 90-day window.**

- Grain check: `content_id` is already unique per row in this starter slice (30,000 rows, 30,000 unique `content_id`s) — confirmed in the code cell below.
- `client_id` is not the row grain; it groups pages by client (32 clients) and exists for grouped train/test splits later, per `skills/flyrank/flyrank-data/SKILL.md` ("Use client_id for grouped train/test splits").
- IDs are pseudonyms only — never features, per the same skill file.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.

print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())
print("Unique client_id:", df["client_id"].nunique())
print("One row per content page?", len(df) == df["content_id"].nunique())
print()

unit_of_analysis = df[[
    "content_id", "client_id", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "engagement_rate", "word_count", "trend_direction", "is_declining_label"
]]
display(unit_of_analysis.head())


Rows: 30000
Unique content_id: 30000
Unique client_id: 32
One row per content page? True



,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,187,20,3803,29,0.76,10.6,5.88,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,445,25,15320,7,0.05,20.3,0.00,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,141,20,12581,11,0.09,36.5,0.00,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,463,22,11751,58,0.49,6.2,1.28,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,263,14,19140,24,0.13,44.0,0.00,2803.0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**No single column reliably marks a declining page — and a plausible one-line rule actually does worse than guessing.**

- Correlation check (code below): every candidate signal — impressions, CTR, position, engagement rate, scroll rate, word count, age, days since update — correlates with `is_declining_label` at 0.16 or weaker. None is a usable if-statement on its own.
- Concrete proof: a naive, "obvious" rule — *flag every page older than the median age as decline-risk* — gets **0.456 precision**. That is *worse* than the 0.542 base rate from random guessing. A rule built on gut instinct about one signal (age) actively hurts here.
- Why this happens: decline looks tangled up with several weak, only-loosely-related signals at once (a bit older, a bit stale, a bit lower CTR, a bit weaker engagement) rather than one strong cause. That's exactly the case the framing skill describes: "ML earns its place only when the pattern is real but too messy to write by hand — many signals, tangled." A model that can weigh and combine several weak signals has a real shot; a single if-statement doesn't.
- One-paragraph frame, filled in: *For the SEO content team, deciding which of ~30,000 pages to review first, we will build a ranked queue from 90-day search and content signals, scoring pages by decline risk measured by Precision@50 against a 0.542 base rate. A wrong call costs wasted editor hours (false positive) or continued silent traffic loss (false negative). A plain rule isn't enough because no single signal correlates above 0.16 with the label, and the one intuitive rule I tested underperforms random guessing. We will claim only observed, directional, decision-support results — not causal or future-guaranteed ones.*


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.

candidates = ["impressions_90d", "ctr", "avg_position", "engagement_rate",
              "scroll_rate", "word_count", "content_age_days", "days_since_last_update"]

print("Correlation of each single signal with is_declining_label:")
for c in candidates:
    print(f"  {c:<24s} {df[c].corr(df['is_declining_label']):+.3f}")

print()
median_age = df["content_age_days"].median()
naive_flag = (df["content_age_days"] > median_age)
naive_precision = (naive_flag & (df["is_declining_label"] == 1)).sum() / naive_flag.sum()
print(f"Naive rule 'age > median ({median_age:.0f} days) = decline risk':")
print(f"  Precision: {naive_precision:.3f}  (worse than the {base_rate:.3f} base rate)")


Correlation of each single signal with is_declining_label:
  impressions_90d          -0.018
  ctr                      -0.062
  avg_position             -0.029
  engagement_rate          -0.013
  scroll_rate              -0.003
  word_count               +0.090
  content_age_days         -0.164
  days_since_last_update   +0.081

Naive rule 'age > median (236 days) = decline risk':
  Precision: 0.456  (worse than the 0.542 base rate)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
